[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github.com/MLinApp-polito/mla-prj-23-project-am04_group-am01/blob/main/defect_detection.ipynb)

TODO: fix the button

# PBF Defect Detection

This notebook covers data loading, training, validation, and inference for detecting defects in Powder Bed Fusion images using a fine-tuned CNN.

## Clone GithHub repo

In [ ]:
!rm -rf mla_project/

In [ ]:
import os

if not os.path.exists("/content/mla-prj-23-project-am04_group-am01") and not os.path.exists("/content/mla_project"):
  # DON'T SHARE THE PERSONAL ACCESS TOKEN

  # change the name of the branch here as needed
  !git clone -b gan https://***REMOVED-GITHUB-TOKEN***@github.com/MLinApp-polito/mla-prj-23-project-am04_group-am01.git

  # Rename folder for simplicity
  !mv /content/mla-prj-23-project-am04_group-am01 /content/mla_project

!cd /content/mla_project && git pull

Cloning into 'mla-prj-23-project-am04_group-am01'...
remote: Enumerating objects: 2742, done.
remote: Counting objects: 100% (377/377), done.
remote: Compressing objects: 100% (189/189), done.
remote: Total 2742 (delta 227), reused 294 (delta 167), pack-reused 2365 (from 3)
Receiving objects: 100% (2742/2742), 1.83 GiB | 32.57 MiB/s, done.
Resolving deltas: 100% (1279/1279), done.
Already up to date.


## Install Dependencies

In [ ]:
!pip install torch torchvision matplotlib tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 123.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 44.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

## Imports

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# Original dataset

## Dataset mean and std

In [ ]:
!python /content/mla_project/src/data_loader.py --data-dir /content/mla_project/images --compute-stats

Images shape:  torch.Size([16, 1, 1024, 1280])
Images shape:  torch.Size([16, 1, 1024, 1280])
Images shape:  torch.Size([16, 1, 1024, 1280])
Images shape:  torch.Size([16, 1, 1024, 1280])
Images shape:  torch.Size([10, 1, 1024, 1280])
Dataset mean (grayscale): 0.5830
Dataset std (grayscale): 0.2075


## Training - no augmentation

**IMPORTANT:**

- To perform K-Fold cross-validation, set --is_kfold to "True" and specify the number of folds with --k-folds. Example: --is_kfold "True", --k-folds 5

- To perform a single train/val split, set --is_kfold to "False" and specify the validation split ratio with --val-split. Example: --is_kfold "False", --val-split 0.2

- In both cases, to perform also testing, set --test to "True" and specify the test split ratio with --test-split. Example: --test "True", --test-split 0.2

### Define paths and parameters

In [ ]:
data_dir = '/content/mla_project/images'
train_dir = os.path.join(data_dir, 'train')
val_dir   = os.path.join(data_dir, 'val')

# Training params
batch_size = 1
epochs = 10
learning_rate = 1e-3
backbone = 'resnet50'
device = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


### Launch training

In [ ]:
# k-fold cross validation (with test)

!python /content/mla_project/src/train.py \
    --data-dir "{data_dir}" \
    --batch-size {batch_size} \
    --epochs {10} \
    --lr {learning_rate} \
    --backbone {backbone} \
    --num-workers 2 \
    --is_kfold "True" \
    --k-folds 5 \
    --test "True" \
    --test-split 0.2

### Plot Training & Validation Curves

In [ ]:
# plot for cross-validation
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Read logs
logs = pd.read_csv('/content/kfold_logs.csv')

# Group for epoch and get mean and std
grouped = logs.groupby('epoch').agg({
    'train_loss': ['mean', 'std'],
    'val_loss': ['mean', 'std'],
    'train_acc': ['mean', 'std'],
    'val_acc': ['mean', 'std']
}).reset_index()

# Rename columns
grouped.columns = ['epoch',
                   'train_loss_mean', 'train_loss_std',
                   'val_loss_mean', 'val_loss_std',
                   'train_acc_mean', 'train_acc_std',
                   'val_acc_mean', 'val_acc_std']

# Set style
sns.set(style="white", context="notebook")

# LOSS
plt.figure(figsize=(8, 5))
plt.plot(grouped['epoch'], grouped['train_loss_mean'], label='Train Loss', color='blue')
plt.fill_between(grouped['epoch'],
                 grouped['train_loss_mean'] - grouped['train_loss_std'],
                 grouped['train_loss_mean'] + grouped['train_loss_std'],
                 color='blue', alpha=0.2)

plt.plot(grouped['epoch'], grouped['val_loss_mean'], label='Val Loss', color='orange')
plt.fill_between(grouped['epoch'],
                 grouped['val_loss_mean'] - grouped['val_loss_std'],
                 grouped['val_loss_mean'] + grouped['val_loss_std'],
                 color='orange', alpha=0.2)

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss (mean ± std)')
plt.legend()
sns.despine()
plt.tight_layout()
plt.show()

# ACCURACY
plt.figure(figsize=(8, 5))
plt.plot(grouped['epoch'], grouped['train_acc_mean'], label='Train Accuracy', color='green')
plt.fill_between(grouped['epoch'],
                 grouped['train_acc_mean'] - grouped['train_acc_std'],
                 grouped['train_acc_mean'] + grouped['train_acc_std'],
                 color='green', alpha=0.2)

plt.plot(grouped['epoch'], grouped['val_acc_mean'], label='Val Accuracy', color='red')
plt.fill_between(grouped['epoch'],
                 grouped['val_acc_mean'] - grouped['val_acc_std'],
                 grouped['val_acc_mean'] + grouped['val_acc_std'],
                 color='red', alpha=0.2)

plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy (mean ± std)')
plt.legend()
sns.despine()
plt.tight_layout()
plt.show()


## Training - basic augmentations (simple transformations)

In [ ]:
data_dir = '/content/mla_project/images'
train_dir = os.path.join(data_dir, 'train')
val_dir   = os.path.join(data_dir, 'val')

# Training params
batch_size = 1
epochs = 10
learning_rate = 1e-3
backbone = 'resnet50'
device = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# k-fold cross validation (with test) with augmented data

!python /content/mla_project/src/train.py \
    --data-dir "{data_dir}" \
    --batch-size {batch_size} \
    --epochs {10} \
    --lr {learning_rate} \
    --backbone {backbone} \
    --num-workers 2 \
    --is_kfold "True" \
    --k-folds 5 \
    --test "True" \
    --test-split 0.2 \
    --aug "True"

In [ ]:
# plot for cross-validation
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Read logs
logs = pd.read_csv('/content/kfold_logs.csv')

# Group for epoch and get mean and std
grouped = logs.groupby('epoch').agg({
    'train_loss': ['mean', 'std'],
    'val_loss': ['mean', 'std'],
    'train_acc': ['mean', 'std'],
    'val_acc': ['mean', 'std']
}).reset_index()

# Rename columns
grouped.columns = ['epoch',
                   'train_loss_mean', 'train_loss_std',
                   'val_loss_mean', 'val_loss_std',
                   'train_acc_mean', 'train_acc_std',
                   'val_acc_mean', 'val_acc_std']

# Set style
sns.set(style="white", context="notebook")

# LOSS
plt.figure(figsize=(8, 5))
plt.plot(grouped['epoch'], grouped['train_loss_mean'], label='Train Loss', color='blue')
plt.fill_between(grouped['epoch'],
                 grouped['train_loss_mean'] - grouped['train_loss_std'],
                 grouped['train_loss_mean'] + grouped['train_loss_std'],
                 color='blue', alpha=0.2)

plt.plot(grouped['epoch'], grouped['val_loss_mean'], label='Val Loss', color='orange')
plt.fill_between(grouped['epoch'],
                 grouped['val_loss_mean'] - grouped['val_loss_std'],
                 grouped['val_loss_mean'] + grouped['val_loss_std'],
                 color='orange', alpha=0.2)

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss (mean ± std)')
plt.legend()
sns.despine()
plt.tight_layout()
plt.show()

# ACCURACY
plt.figure(figsize=(8, 5))
plt.plot(grouped['epoch'], grouped['train_acc_mean'], label='Train Accuracy', color='green')
plt.fill_between(grouped['epoch'],
                 grouped['train_acc_mean'] - grouped['train_acc_std'],
                 grouped['train_acc_mean'] + grouped['train_acc_std'],
                 color='green', alpha=0.2)

plt.plot(grouped['epoch'], grouped['val_acc_mean'], label='Val Accuracy', color='red')
plt.fill_between(grouped['epoch'],
                 grouped['val_acc_mean'] - grouped['val_acc_std'],
                 grouped['val_acc_mean'] + grouped['val_acc_std'],
                 color='red', alpha=0.2)

plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy (mean ± std)')
plt.legend()
sns.despine()
plt.tight_layout()
plt.show()

# Generative Adversarial Networks

## Training - GANs

In [4]:
!python /content/mla_project/external/PyTorch-GAN/implementations/dcgan/dcgan.py \
    --generate_defect "True" \
    --data_dir "/content/mla_project/images/original/Defects" \
    --batch_size 4 \
    --n_epochs 600 \
    --img_size 800 \
    --sample_interval 400 \
    --R1_regularization "True"

Output streaming troncato alle ultime 5000 righe.
[Epoch 183/600] [Batch 7/12] [D loss: 0.095451] [G loss: 1.734912]
[Epoch 183/600] [Batch 8/12] [D loss: 0.279112] [G loss: 3.606738]
[Epoch 183/600] [Batch 9/12] [D loss: 0.111524] [G loss: 0.852363]
[Epoch 183/600] [Batch 10/12] [D loss: 0.263180] [G loss: 3.770637]
[Epoch 183/600] [Batch 11/12] [D loss: 0.258632] [G loss: 3.391017]
[Epoch 184/600] [Batch 0/12] [D loss: 0.183349] [G loss: 1.343366]
[Epoch 184/600] [Batch 1/12] [D loss: 0.227570] [G loss: 2.485896]
[Epoch 184/600] [Batch 2/12] [D loss: 0.142535] [G loss: 2.801240]
[Epoch 184/600] [Batch 3/12] [D loss: 0.142817] [G loss: 2.998064]
[Epoch 184/600] [Batch 4/12] [D loss: 0.034675] [G loss: 6.410554]
[Epoch 184/600] [Batch 5/12] [D loss: 0.280626] [G loss: 4.630414]
[Epoch 184/600] [Batch 6/12] [D loss: 0.157393] [G loss: 1.661522]
[Epoch 184/600] [Batch 7/12] [D loss: 0.105582] [G loss: 2.521539]
[Epoch 184/600] [Batch 8/12] [D loss: 0.738090] [G loss: 0.210007]
[Epoch 184

### Generate new images using GANs

In [5]:
!python /content/mla_project/external/PyTorch-GAN/implementations/dcgan/generate.py \
      --model_path "/content/saved_models/generator_epoch_599.pth" \
      --img_size 800 \
      --num_images 5 \
      --output_dir "/content/generated_images"

Generated 5 images in /content/generated_images


In [6]:
!zip -r /content/generated_images.zip /content/generated_images

  adding: content/generated_images/ (stored 0%)
  adding: content/generated_images/image_4.png (deflated 0%)
  adding: content/generated_images/image_3.png (deflated 0%)
  adding: content/generated_images/image_2.png (deflated 0%)
  adding: content/generated_images/image_0.png (deflated 0%)
  adding: content/generated_images/image_1.png (deflated 0%)


In [7]:
!python /content/mla_project/external/PyTorch-GAN/implementations/dcgan/generate.py \
      --model_path "/content/saved_models/generator_epoch_599.pth" \
      --img_size 800 \
      --num_images 5 \
      --output_dir "/content/generated_images_2"
!zip -r /content/generated_images_2.zip /content/generated_images_2

Generated 5 images in /content/generated_images_2
  adding: content/generated_images_2/ (stored 0%)
  adding: content/generated_images_2/image_4.png (deflated 0%)
  adding: content/generated_images_2/image_3.png (deflated 0%)
  adding: content/generated_images_2/image_2.png (deflated 0%)
  adding: content/generated_images_2/image_0.png (deflated 0%)
  adding: content/generated_images_2/image_1.png (deflated 0%)


### Upload model to HuggingFace

Token: ***REMOVED-HUGGINGFACE-TOKEN***

In [10]:
from huggingface_hub import login, create_repo, upload_file

# Effettua il login (ti verrà richiesto di inserire il token)
login()

# Carica un file nel repository
upload_file(
    path_or_fileobj="/content/saved_models/generator_epoch_599.pth",
    path_in_repo="Experiment7_generator_epoch_599.pth",
    repo_id="MLinAppl/gan",
    repo_type="model"
)

generator_epoch_599.pth:   0%|          | 0.00/2.64G [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/MLinAppl/gan/commit/26c0a8cd7f14a2ad0c193cb435f7542ec06449cf', commit_message='Upload Experiment7_generator_epoch_599.pth with huggingface_hub', commit_description='', oid='26c0a8cd7f14a2ad0c193cb435f7542ec06449cf', pr_url=None, repo_url=RepoUrl('https://huggingface.co/MLinAppl/gan', endpoint='https://huggingface.co', repo_type='model', repo_id='MLinAppl/gan'), pr_revision=None, pr_num=None)